<img src="../../img/backdrop-wh.png" alt="Drawing" style="width: 300px;"/>

# Optional: Running a Small Language Model Locally

* * *


<div class="alert alert-success">

### Learning Objectives

* Describe the difference between a hosted LLM and a local LLM.
* Recognize that models have material constraints: file size, memory, speed, context limits, and hardware dependence.
* Experiment with generation settings such as temperature and maximum output length.
* Compare a small local model's interpretive output with a hosted model's output.
* Reflect on the tradeoff between control, privacy, access, fluency, and performance.
* Understand that running a model locally does not fully "open the black box," but it does make parts of the infrastructure more visible.

</div>

### Icons Used in This Notebook
🔔 **Question**: A quick question to help you understand what's going on.<br>
💡 **Tip**: How to do something a bit more efficiently or effectively.<br>
⚠️ **Warning:** Heads-up about tricky stuff or common mistakes.<br>
💭 **Reflection**: Reflecting on ethical implications, biases, and social impact in data science.<br>

### Sections
1. [Orientation](#orient)
2. [Local vs. Hosted Models](#compare)
3. [Installation Options](#install)
4. [Pull or Select a Small Model](#model)
5. [First Local Generation](#first)
6. [Generation Settings](#settings)
7. [Apply the Local Model to a Reddit Excerpt](#apply)
8. [Compare with a Hosted Model](#hosted)
9. [What Local Models Reveal and Conceal](#reveal)
10. [Optional: Connection to Course Concepts](#concepts)
11. [Exit Ticket](#exit)


<a id='orient'></a>

# 0. Orientation

**Estimated time:** 60–120 minutes, depending on installation | **Status:** Optional

This notebook is an optional extension. It is not required, and installation success is not the basis for credit.

If you cannot install Ollama or if your machine does not have sufficient memory to run a local model, you can still read through the notebook, compare the outputs provided here with hosted model outputs, and write a reflection based on what you observe. A *read-only path* is noted at each stage.

---

This lab does not fully open the black box. Instead, it makes the model more material.

You will see that a language model is not only a chat interface but also a file, a process, a set of settings, a hardware burden, and a limited generator of text.

What you will observe:

- How large a model file is.
- How long inference takes without a company's data center behind it.
- How generation settings change output.
- How a small model's interpretive performance compares with a hosted frontier model.
- What is more visible locally, and what remains opaque regardless.

This model may be much weaker than ChatGPT, Claude, or Gemini. **That weakness is part of the lesson.**


<a id='compare'></a>

# 1. Local vs. Hosted Models

Before you install anything, take a moment to understand what you are comparing.

| | Hosted model (e.g., Gemini, ChatGPT) | Local model (e.g., via Ollama) |
|---|---|---|
| **Runs on** | Company servers | Your machine |
| **Strength** | Usually stronger and more fluent | Usually smaller and weaker |
| **Privacy** | Prompts sent to external service | Stays on your machine |
| **Infrastructure** | Hidden behind a polished interface | More visible: file, memory, latency |
| **Control** | Limited (model choice, settings) | More: model, settings, offline use |
| **Safety layers** | Often includes product-specific behavior | Varies by model and configuration |
| **Transparency** | Black box | Still largely a black box, but materially different |

Neither option is fully transparent. Local models let you see more of the *infrastructure* — the file size, the hardware limits, the latency — without revealing training data, internal representations, or the reasons for specific outputs.

### Student Action: Pre-Lab Prediction

*Replace this text with your answer.*

**Before running anything: What do you expect a small local model to do worse than a hosted model?**

**Is there anything you expect it to do better?**

💭 **Reflection:** What does it mean for a model to be "better"? Better at what? For whom?


<a id='install'></a>

# 2. Installation Options

Choose **one** of the two options below. You do not need both.

---

## Option A: Ollama (Recommended)

Ollama is a command-line tool that lets you download and run language models locally. It is the recommended path for this notebook because it integrates cleanly with Python via a REST API that we can call using `requests` — a package already installed in your course environment.

**Installation steps (high level):**

1. Go to [ollama.com](https://ollama.com) and download the installer for your OS (macOS, Windows, Linux).
2. Run the installer.
3. Open a terminal and verify installation:
   ```
   ollama --version
   ```
4. Pull a small model (we will do this in Section 3).
5. Ollama runs as a background service on `http://localhost:11434`. Leave it running while you work in this notebook.

💡 **Tip (macOS):** After installing, Ollama may appear in your menu bar. You can start and stop it from there.

💡 **Tip (Windows):** If you installed Ollama but the notebook cannot connect, open a new terminal and type `ollama serve` to start the server manually.

---

## Option B: LM Studio (Alternative)

LM Studio provides a graphical interface for downloading and chatting with local models. It may be easier if you prefer not to use the terminal.

**To use LM Studio with this notebook:**

1. Download LM Studio from [lmstudio.ai](https://lmstudio.ai).
2. Download a small model inside the app (search for a 1B–3B parameter quantized model).
3. In the "Local Server" tab, enable the server. It typically runs on `http://localhost:1234`.
4. Update the `OLLAMA_BASE_URL` variable in Section 4 to `http://localhost:1234`.

⚠️ **Warning:** LM Studio's API format is slightly different from Ollama's. The code in this notebook targets the Ollama API. If you use LM Studio, you may need to adjust the request format.

---

### Student Action: Tool Choice

*Replace this text with your answer.*

**Which tool are you using (Ollama or LM Studio)?**

**Did installation or setup change how you think about AI systems in any way?**


<a id='model'></a>

# 3. Pull or Select a Small Model

### Choosing a model

Not all models run on all machines. The right choice depends on your available RAM and storage.

| Model | Size on disk | RAM needed | Notes |
|---|---|---|---|
| `llama3.2:1b` | ~1.3 GB | ~4 GB RAM | Recommended default. Very fast, weaker quality. |
| `qwen2.5:1.5b` | ~1.0 GB | ~4 GB RAM | Good alternative, strong at reasoning. |
| `llama3.2:3b` | ~2.0 GB | ~6 GB RAM | Better quality, still accessible on most laptops. |
| `phi3:mini` | ~2.3 GB | ~6 GB RAM | Microsoft's small model, often capable for its size. |
| `mistral:7b` | ~4.1 GB | ~8 GB RAM | Much better quality, but requires more hardware. |

**Recommended default:** `llama3.2:1b`. It runs on almost any modern laptop and downloads quickly.

**To pull a model, run this in your terminal (not in the notebook):**

```
ollama pull llama3.2:1b
```

This downloads the model file once. After that it is cached locally.

💡 **Tip:** Run `ollama list` in your terminal to see which models you already have downloaded.

### What "quantized" means

Most models available through Ollama are *quantized* — compressed versions that take less space and memory. Quantization reduces the precision of the model's weights. This means the model runs more easily on consumer hardware, but it may lose some quality compared to the full-precision version. Quantization is one reason local models are often weaker than hosted frontier models, which typically run on full-precision or high-precision weights across many GPUs.


In [ ]:
import requests

# ── CONFIGURE YOUR LOCAL MODEL ────────────────────────────────────────────────
MODEL_NAME = 'llama3.2:1b'        # change to whichever model you pulled
OLLAMA_BASE_URL = 'http://localhost:11434'

def check_ollama():
    """Check whether the Ollama server is running and list available models."""
    try:
        r = requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=5)
        models = [m['name'] for m in r.json().get('models', [])]
        if models:
            print("Ollama is running. Available models:")
            for m in models:
                print(f"  {m}")
        else:
            print("Ollama is running but no models are downloaded yet.")
            print(f"Run this in your terminal:  ollama pull {MODEL_NAME}")
        return True
    except Exception as e:
        print(f"Cannot connect to Ollama: {e}")
        print("Make sure Ollama is installed and running.")
        print("If it is installed, open a terminal and type: ollama serve")
        return False

check_ollama()

### Student Action: Record Your Model

*Replace this text with your answer.*

**What model did you choose?**

**Approximately how large is the model file?** (Run `ollama list` in your terminal to see size.)

**Did it run smoothly, slowly, or not at all?**

**What does this suggest about the material requirements of language modeling?**

---

📖 **Read-only path:** If you cannot run Ollama, read this section and the outputs below, then skip to Section 7 to compare model outputs using screenshots or text examples from a hosted model.


In [ ]:
def query_ollama(prompt, model=MODEL_NAME, temperature=0.7, max_tokens=300):
    """
    Send a prompt to the local Ollama model and return the response text.

    Parameters
    ----------
    prompt : str
        The text prompt to send.
    model : str
        The model name to use (must be pulled via `ollama pull`).
    temperature : float
        Controls randomness. Lower = more predictable, higher = more varied.
    max_tokens : int
        Maximum number of tokens to generate.

    Returns
    -------
    str
        The model's response text.
    """
    payload = {
        'model': model,
        'prompt': prompt,
        'stream': False,
        'options': {
            'temperature': temperature,
            'num_predict': max_tokens,
        }
    }
    try:
        response = requests.post(
            f'{OLLAMA_BASE_URL}/api/generate',
            json=payload,
            timeout=120
        )
        response.raise_for_status()
        return response.json()['response']
    except requests.exceptions.ConnectionError:
        return "[ERROR] Could not connect to Ollama. Is the server running?"
    except requests.exceptions.Timeout:
        return "[ERROR] Request timed out. The model may need more time on your hardware."
    except Exception as e:
        return f"[ERROR] {e}"

print("Helper function loaded.")
print(f"Model: {MODEL_NAME} | Base URL: {OLLAMA_BASE_URL}")

<a id='first'></a>

# 4. First Local Generation

Send a conceptual, course-relevant prompt to the local model. This gives you a baseline sense of its capability before asking it to analyze real data.

🔔 **Question:** What do you notice about response time? About fluency? About specificity?


In [ ]:
import time

prompt_1 = (
    "In two short paragraphs, explain what topic modeling is and "
    "why a researcher might use it to study Reddit posts."
)

print(f"Prompt: {prompt_1}\n")
print("Generating...\n")

start = time.time()
response_1 = query_ollama(prompt_1)
elapsed = time.time() - start

print(f"Response ({elapsed:.1f}s):\n")
print(response_1)

In [ ]:
# Try a second prompt about a course concept
prompt_2 = (
    "What does it mean to interpret a text hermeneutically? "
    "Give a concrete example involving a Reddit post. Answer in 3–4 sentences."
)

print(f"Prompt: {prompt_2}\n")
print("Generating...\n")

start = time.time()
response_2 = query_ollama(prompt_2)
elapsed = time.time() - start

print(f"Response ({elapsed:.1f}s):\n")
print(response_2)

### Student Action: First Impressions

*Replace this text with your answer.*

**What did you notice about speed?**

**What did you notice about fluency and specificity?**

**Did the model feel different from hosted systems you have used?**

💭 **Reflection:** What does it mean for a model to be "fast" or "slow"? What infrastructure makes hosted models feel instant?


<a id='settings'></a>

# 5. Generation Settings

Language models do not generate text deterministically. They sample from a probability distribution over possible next tokens. Two settings shape that sampling:

**Temperature** controls randomness.
- Low temperature (e.g., 0.1–0.3): the model strongly prefers the most likely next token. Outputs tend to be repetitive, predictable, or formulaic.
- High temperature (e.g., 0.9–1.2): the model spreads probability more evenly, producing more varied and sometimes unexpected output.
- Think of it as: *how willing is the model to take a linguistic risk?*

**`num_predict` (max tokens)** controls how long the output can be.
- Short limits produce truncated or incomplete responses.
- Very long limits on a small model may produce repetition or incoherence.

The same prompt can produce noticeably different outputs at different temperature settings. This is not a bug. It is a fundamental property of how language models generate text — and it has interpretive consequences.


In [ ]:
prompt_settings = (
    "Describe the community norms of r/AmITheAsshole in 3 sentences. "
    "What kinds of stories do people post? What kinds of judgments does the community make?"
)

settings_to_test = [
    {'label': 'Low temperature (0.1) — predictable', 'temperature': 0.1},
    {'label': 'Medium temperature (0.7) — default',  'temperature': 0.7},
    {'label': 'High temperature (1.2) — variable',   'temperature': 1.2},
]

print(f"Prompt: {prompt_settings}\n")
print("=" * 60)

for cfg in settings_to_test:
    label = cfg['label']
    temp  = cfg['temperature']
    print(f"\n--- {label} ---\n")
    start = time.time()
    out = query_ollama(prompt_settings, temperature=temp, max_tokens=200)
    elapsed = time.time() - start
    print(f"{out}\n[{elapsed:.1f}s]")
    print("-" * 60)

### Student Action: Settings Observations

*Replace this text with your answer.*

**What changed between temperature settings?**

**Did the model become more useful, more repetitive, more creative, or less reliable?**

💭 **Reflection:** If temperature is a parameter that shapes the *randomness* of a model's interpretation, what does this say about machine interpretation as a concept? Is a human reader's interpretation similarly adjustable?


<a id='apply'></a>

# 6. Apply the Local Model to a Reddit Excerpt

Now ask the local model to analyze one real post from your course dataset.

**Important constraints for small models:**
- Keep your prompt short. Small local models have limited context windows (often 2,000–8,000 tokens) and struggle with long inputs.
- Focus on a single post, not a summary of the whole dataset.
- Ask a focused question, not an open-ended one.

The goal is not to get a comprehensive analysis. The goal is to observe *how this model interprets a specific text* — and compare it with what you would expect from a hosted model.


In [ ]:
import pandas as pd

# Load the dataset
try:
    df = pd.read_csv('../../data/aita_pp.csv')
except FileNotFoundError:
    df = pd.read_csv('../../data/aita_top_submissions.csv')

df = df.dropna(subset=['selftext'])

# Select one short-to-medium post (300–600 words is a good target for small models)
word_counts = df['selftext'].apply(lambda x: len(str(x).split()))
suitable = df[(word_counts >= 100) & (word_counts <= 400)].reset_index(drop=True)

# Display a sample to choose from
for i in range(3):
    post = suitable.iloc[i]
    wc = len(str(post['selftext']).split())
    print(f"[{i}] {wc} words | Flair: {post.get('flair_text', 'N/A')}")
    print(f"    {str(post['selftext'])[:200]}...")
    print()

In [ ]:
# Choose one post by index from the display above
chosen_idx = 0
chosen_post = suitable.iloc[chosen_idx]['selftext']

print(f"Selected post ({len(chosen_post.split())} words):\n")
print(chosen_post)

In [ ]:
# ── LOCAL MODEL ANALYSIS ─────────────────────────────────────────────────────
# Construct a focused analytical prompt
analysis_prompt = (
    "Read the following Reddit post from r/AmITheAsshole carefully. "
    "In 3–4 sentences, identify: (1) the central conflict, "
    "(2) the implicit norm or value the poster is appealing to, "
    "and (3) the rhetorical strategy they use to frame themselves sympathetically.\n\n"
    f"Post:\n{chosen_post[:800]}"  # truncate for model context
)

print("Sending to local model...\n")
start = time.time()
local_response = query_ollama(analysis_prompt, temperature=0.5, max_tokens=300)
elapsed = time.time() - start

print(f"Local model response ({elapsed:.1f}s):\n")
print(local_response)

### Student Action: Local Analysis Notes

*Replace this text with your answer.*

**What did the local model notice?**

**What did it miss or oversimplify?**

**Did it impose a generic interpretation, moral frame, or summary pattern?**

💭 **Reflection:** A small local model has seen far less text during training than a frontier model. How might training data volume or diversity affect interpretive performance?


<a id='hosted'></a>

# 7. Compare with a Hosted Model

Send the same post and the same prompt to a hosted model — Gemini, ChatGPT, Claude, or another tool.

You do not need an API key for this step. Use the browser interface.

Copy the `analysis_prompt` text from Section 6 into the hosted model's chat interface and paste the response below.

### Things to look for

| Dimension | Questions to ask |
|---|---|
| **Fluency** | Which response is more grammatically coherent and readable? |
| **Specificity** | Which response refers more precisely to the actual text of the post? |
| **Caution / safety** | Did one model hedge more, add disclaimers, or refuse anything? |
| **Moral framing** | Did one model introduce a stronger moral judgment? |
| **Verbosity** | Which model produced more text? Was that extra text useful? |
| **Interpretive imagination** | Which model offered a more surprising or nuanced reading? |


In [ ]:
# Print the prompt so you can copy it easily
print("=== Prompt to paste into hosted model ===\n")
print(analysis_prompt)
print("\n=== End of prompt ===")

### Student Action: Paste Hosted Model Response

*Replace this text with the hosted model's response.*

**Hosted model used (Gemini / ChatGPT / Claude / other):**

**Response:**

> [paste here]

---

### Student Action: Comparison

*Replace this text with your comparison.*

**Which was more fluent?**

**Which was more specific to the actual text?**

**Which was more cautious or added more hedges?**

**Which introduced more interpretive assumptions?**

**Which felt more useful for humanities-style close reading?**

💭 **Reflection:** How did the local model's interpretation differ from the hosted model's? What does this difference reveal about what "interpretation" means in computational terms?


<a id='reveal'></a>

# 8. What Local Models Reveal and Conceal

Running a model locally makes some things more visible. It does not make the model transparent.

### What running locally reveals

| Feature | What you can observe |
|---|---|
| **Model size** | The download is a real file with a real size. |
| **Hardware dependence** | Inference speed varies with your CPU, GPU, and RAM. |
| **Latency** | You feel the compute cost directly — no data center absorbing it for you. |
| **Context limits** | Long prompts can cause errors or truncation. |
| **Generation settings** | You control temperature and output length and can see what they do. |
| **Model comparison** | You can swap models and compare outputs side by side. |

### What running locally still conceals

| Feature | Why it remains opaque |
|---|---|
| **Training data** | The model file does not contain the training corpus. |
| **Training process** | RLHF, instruction tuning, and fine-tuning choices are not visible. |
| **Internal representations** | Weight matrices are not human-readable. |
| **Reasons for outputs** | You cannot determine why a specific word was chosen. |
| **Alignment decisions** | Safety and refusal behaviors were built in during training. |
| **Institutional decisions** | Who chose the training data, the RLHF objectives, the release policy. |

The model is more material when it runs on your machine. It is not more legible.

### Student Action: After-Lab Reflection

*Replace this text with your answer.*

**After this lab, what feels less mysterious about LLMs?**

**What remains opaque, even after running a model locally?**


<a id='concepts'></a>

# 9. Optional: Connection to Course Concepts

This section is optional. If you have time, write a 200–300 word response connecting this lab to the course's broader themes of mediation, interpretation, and digital hermeneutics.

Consider:

- **Mediation:** Every interaction with an LLM is mediated — by the interface, the API, the model file, the hardware, the training data, and the alignment process. What does local inference change about that mediation?
- **Distance:** Hermeneutics talks about the productive distance between a text and its reader. Is a local model a closer or more distant reader? What does that even mean?
- **Materiality:** The model is a file. It has a size. It requires RAM. It heats up your laptop. Does this materiality change how you think about machine "reading"?
- **Opacity:** The model is not interpretable even locally. What are the limits of transparency as a goal for AI systems?

💭 **Reflection:** How does running a model locally change your understanding of machine interpretation? Does it make the model seem more like a tool, a text, an interlocutor, an infrastructure, or something else?

### Student Action: Write 200–300 words (optional)

*Replace this text with your response.*


<a id='exit'></a>

# 10. Exit Ticket

Answer the four questions below to complete the lab.

---

**1. What was the most concrete thing this lab helped you understand about LLMs?**

*Replace this text with your answer.*

---

**2. What was the biggest technical barrier you encountered?**

*Replace this text with your answer.*

---

**3. Would you use a local model for serious interpretive research? Why or why not?**

*Replace this text with your answer.*

---

**4. What would you still want to know about how these models work?**

*Replace this text with your answer.*

---

### Optional Extension: Extra Credit Submission

If your instructor offers extra credit for this lab, submit **400–600 words** addressing:

> What did running or attempting to run a local model reveal about the material conditions of AI? Compare one local-model output with one hosted-model output. What did the comparison show about fluency, interpretation, opacity, and control?

---

<div class="alert alert-success">

## ❗ Key Points

* A local model is a file with a size, a runtime, a context limit, and hardware requirements. These are visible locally in ways they are not behind a hosted interface.
* Local models are typically weaker than hosted frontier models. That weakness is part of the lesson.
* Running a model locally makes the infrastructure more material, but not more legible. Training data, weights, alignment decisions, and reasons for outputs remain opaque.
* Temperature controls randomness in generation — not creativity in any human sense.
* The comparison between a small local model and a hosted model reveals something about what scale, data, and infrastructure contribute to "interpretive performance."

</div>
